# Slime Mold Sweep — Average Performance by Agent Count

This notebook loads the **low-agent** (500 → 8 000) and **high-agent** (10 000 → 160 000) slime-mold benchmark suites, combines them, and plots the **average frame execution time** for each compute method as agent count increases.

In [ ]:
import sys, pathlib

# Ensure src/ is importable regardless of CWD
PROJECT_ROOT = pathlib.Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_all_suites
from src.plot_style import apply_style, save_figure, get_method_color
from src.constants import METHOD_ORDER, METHOD_LABELS, METHOD_COLORS

In [ ]:
apply_style()

## 1 — Load & Combine Data

In [ ]:
df = load_all_suites(PROJECT_ROOT / "raw-data")
print(f"Loaded {len(df)} runs across suites: {df['suite'].unique().tolist()}")
df.head()

## 2 — Filter to CPU-render runs (primary comparison)

We compare compute methods using the **CPU render** path so that rendering overhead is held constant. WebGPU GPU-render is shown separately.

In [ ]:
import pandas as pd

# --- CPU-render runs for JS, WASM, WebWorkers ---
cpu_df = df[df["renderMode"] == "cpu"].copy()

# For WebWorkers: keep only the best (lowest avg) worker count per agent count
ww = cpu_df[cpu_df["method"] == "WebWorkers"]
best_ww = ww.loc[ww.groupby("agentCount")["avgExecutionMs"].idxmin()]

# For WebAssembly: keep only the best WASM mode per agent count
wa = cpu_df[cpu_df["method"] == "WebAssembly"]
best_wa = wa.loc[wa.groupby("agentCount")["avgExecutionMs"].idxmin()]

# JavaScript (no sub-variants)
js = cpu_df[cpu_df["method"] == "JavaScript"]

# WebGPU with CPU render (compute on GPU, render on CPU)
gpu_cpu = cpu_df[cpu_df["method"] == "WebGPU"]

# WebGPU with GPU render
gpu_gpu = df[(df["method"] == "WebGPU") & (df["renderMode"] == "gpu")]

plot_df = pd.concat([js, best_ww, best_wa, gpu_cpu, gpu_gpu], ignore_index=True)
plot_df = plot_df.sort_values("agentCount")
print(f"Plot data: {len(plot_df)} rows")
plot_df[["method", "renderMode", "agentCount", "avgExecutionMs"]].head(15)

## 3 — Plot: Average Frame Time vs Agent Count

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

fig, ax = plt.subplots(figsize=(10, 6))

# Define line groups
groups = [
    ("JavaScript",   "cpu", "-",  "o", METHOD_LABELS["JavaScript"]),
    ("WebWorkers",   "cpu", "-",  "s", f"{METHOD_LABELS['WebWorkers']} (best workers)"),
    ("WebAssembly",  "cpu", "-",  "^", f"{METHOD_LABELS['WebAssembly']} (best mode)"),
    ("WebGPU",       "cpu", "-",  "D", f"{METHOD_LABELS['WebGPU']} (CPU render)"),
    ("WebGPU",       "gpu", "--", "D", f"{METHOD_LABELS['WebGPU']} (GPU render)"),
]

for method, render, ls, marker, label in groups:
    subset = plot_df[(plot_df["method"] == method) & (plot_df["renderMode"] == render)]
    if subset.empty:
        continue
    ax.plot(
        subset["agentCount"],
        subset["avgExecutionMs"],
        linestyle=ls,
        marker=marker,
        color=get_method_color(method),
        label=label,
        alpha=0.9,
    )

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Agent Count (N)")
ax.set_ylabel("Average Frame Time (ms)")
ax.set_title("Slime Mold — Average Frame Execution Time vs. Agent Count")
ax.legend(loc="upper left")
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda y, _: f"{y:.1f}"))

save_figure(fig, "slime_sweep_avg_frame_time")
plt.show()